<a href="https://colab.research.google.com/github/jirka73911/geopolars_notebook/blob/main/geopolars_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semestrální práce: ukázka zpracování prostorových dat pomocí Polars a GeoPolars

Cílem této práce je představit knihovnu Polars jako alternativu pro práci s prostorovými daty a ukázat hlavní rozdíly oproti knihovně Pandas.

## 1. Co je Polars?

**Polars** je open-source knihovna pro práci s tabulkovými daty (DataFrame), dostupná mimo jiné pro Python. Původně psána pro jazyk Rust, následně portována do dalších jazyků, jako je třeba Python.

Je určena především pro načítání, filtrování, transformaci a agregaci větších datových tabulek.

### Rozdíly oproti Pandas

Pandas patří mezi nejpoužívanější knihovny pro analýzu dat v Pythonu. Polars řeší podobné úlohy, ale má jinou architekturu a trochu odlišný způsob zápisu.

Snaží se totiž o vyšší rychlost a efektivitu. K tomu využívá například rozdělení operací mezi všechny jádra procesoru. Na druhou stranu, v rámci snahy o udržení lehkosti (co se týká závislostí), podporuje méně formátů než Pandas.

Někdy je mezi těmito knihovnami nutné data v průběhu skriptu převádět - třeba protože jiná knihovna podporuje zatím jen Pandas. K tomu má Polars funkce `to_pandas` a `from_pandas`. Tím se snaží být co nejvíc napojitelná na ekosystémy jiných Python knihoven.

## 2. Kdy zvolit jakou knihovnu?
Když je důležitá široká kompatibilita se zavedenými knihovnami, množství dostupných návodů a inspirací, volím Pandas.
Když pracuji s velkými tabulkami nebo chci využít paralelní zpracování, zvolím Polars. Podle některých programatorů má navíc Polars čitelnější zápis a je obvykle jednodušší na pochopení.

## 3. Co je to GeoPolars?
Projekt, jehož cílem je přidat do Polars možnost pracovat s prostorovými daty. Principielně je to obdoba GeoPandas - taky vedle běžných atributů pracuje rovněž s prostorovou geometrií.

Nevýhodou je, že je zatím ve fázi vývoje a není určena k produkčnímu použití. Veřejnosti je navíc přístupná pouze starší verze z roku 2022. V tomto roce byl vývoj obnoven, ale aktuální verze zatím není veřejná.

Běžnou praxí je dnes využívat hybridní workflow: kombinovat schopnosti a aktuálnost GeoPandas s rychlostí a efektivitou Polars.

## 4. Praktická ukázka
Použijeme data z Natural Earth:
1. Načteme vektorová data států světa.
2. Prohlédneme si atributy a geometrii.
3. Převedeme geometrie do vhodného souřadnicového systému.
4. Pomocí GeoPandas vypočítáme rozlohu polygonů.
5. Výsledek převedeme do běžné tabulky Polars.
6. Pomocí Polars vybereme největší územní jednotky a porovnáme průměrnou rozlohu podle kontinentů.

In [2]:
# ÚKOL 1: Načteme vektorová data států světa (pomocí GeoPandas)

# Import knihoven
import geopandas as gpd
import polars as pl
import pyarrow as pa

# Načtení datové sady
# (Je nezbytné datovou sadu vložit do prostředí Google Colab)
gdf = gpd.read_file('/content/ne_110m_admin_0_countries.shp')

# ÚKOL 2: Prohlédneme si atributy a geometrii
print("Ukázka původních dat v GeoPandas:")
print(gdf.head(10))

Ukázka původních dat v GeoPandas:
        featurecla  scalerank  LABELRANK                   SOVEREIGNT SOV_A3  \
0  Admin-0 country          1          6                         Fiji    FJI   
1  Admin-0 country          1          3  United Republic of Tanzania    TZA   
2  Admin-0 country          1          7               Western Sahara    SAH   
3  Admin-0 country          1          2                       Canada    CAN   
4  Admin-0 country          1          2     United States of America    US1   
5  Admin-0 country          1          3                   Kazakhstan    KA1   
6  Admin-0 country          1          3                   Uzbekistan    UZB   
7  Admin-0 country          1          2             Papua New Guinea    PNG   
8  Admin-0 country          1          2                    Indonesia    IDN   
9  Admin-0 country          1          2                    Argentina    ARG   

   ADM0_DIF  LEVEL               TYPE TLC                        ADMIN  ...  \
0     

Data z Natural Earth jsou uložena ve WGS 84. Abychom mohli přesně spočítat plošnou rozlohu polygonů, musíme mapu převést do zobrazení, které nezkresluje plochy. Pro naše účely využijeme EPSG: 6933

In [3]:
# ÚKOL 3: Převedeme geometrie do vhodného souřadnicového systému (EPSG:6933)
gdf_proj = gdf.to_crs("EPSG:6933")

# ÚKOL 4: Vypočítáme rozlohu polygonů (zatím stále v GeoPandas)
# Výsledek se uloží do nového sloupce v metrech čtverečních
gdf_proj["rozloha_m2"] = gdf_proj.geometry.area

Prostorové výpočty máme za sebou. Nyní geometrii odstraníme a tabulku předáme knihovně Polars.

In [4]:
# ÚKOL 5: Výsledek převedeme do běžné tabulky Polars

# Odstraníme geometrii, zajímají nás už jen textové atributy a rozloha
df_pandas = gdf_proj.drop(columns=["geometry"])

# Bleskový převod z Pandas do Polars
df_polars = pl.from_pandas(df_pandas)

# Úprava dat v Polars (přejmenování sloupce a výpočet km2)
df_polars = (
    df_polars
    .rename({"NAME": "nazev_statu",
            "CONTINENT": "kontinent"})  # Přejmenování sloupce NAME na nazev_statu a CONTINENT na kontinent
    .with_columns(
        (pl.col("rozloha_m2") / 1_000_000).alias("rozloha_km2") # Přepočet na km2
    )
)

print("Data převedena do Polars:")
print(df_polars.select(["nazev_statu", "kontinent", "rozloha_km2"]).head(10))

Data převedena do Polars:
shape: (10, 3)
┌──────────────────────────┬───────────────┬───────────────┐
│ nazev_statu              ┆ kontinent     ┆ rozloha_km2   │
│ ---                      ┆ ---           ┆ ---           │
│ str                      ┆ str           ┆ f64           │
╞══════════════════════════╪═══════════════╪═══════════════╡
│ Fiji                     ┆ Oceania       ┆ 19287.603832  │
│ Tanzania                 ┆ Africa        ┆ 932779.345852 │
│ W. Sahara                ┆ Africa        ┆ 96669.249704  │
│ Canada                   ┆ North America ┆ 1.0038e7      │
│ United States of America ┆ North America ┆ 9.5099e6      │
│ Kazakhstan               ┆ Asia          ┆ 2.7287e6      │
│ Uzbekistan               ┆ Asia          ┆ 461285.460097 │
│ Papua New Guinea         ┆ Oceania       ┆ 464531.61383  │
│ Indonesia                ┆ Asia          ┆ 1.8192e6      │
│ Argentina                ┆ South America ┆ 2.7844e6      │
└──────────────────────────┴───────────────┴

Kdybychom zpracovávali velké objemy dat, byly by to operace jako `group_by` (seskupování) a `sort` (řazení), kde by Polars mělo díky efektivnějšímu využití procesoru nad Pandas excelovat.

My si na našem vzorku států obojí ukážeme.

In [10]:
# ÚKOL 6: Pomocí Polars vybereme největší územní jednotky a porovnáme průměrnou rozlohu kontinentů.

# A) 5 největších zemí světa
nejvetsi_zeme = df_polars.sort("rozloha_km2", descending=True).head(5)
print("5 největších územních jednotek světa")
print(nejvetsi_zeme.select(["nazev_statu", "rozloha_km2"]))

# B) Porovnání průměrné rozlohy státu na kontinentech
kontinenty = (
    df_polars.group_by("kontinent")
    .agg(pl.col("rozloha_km2").mean().alias("prumerna_rozloha_statu_km2"))
    .sort("prumerna_rozloha_statu_km2", descending=True)
)

print("Průměrná rozloha státu podle kontinentů")
print(kontinenty)

5 největších územních jednotek světa
shape: (5, 2)
┌──────────────────────────┬─────────────┐
│ nazev_statu              ┆ rozloha_km2 │
│ ---                      ┆ ---         │
│ str                      ┆ f64         │
╞══════════════════════════╪═════════════╡
│ Russia                   ┆ 1.7021e7    │
│ Antarctica               ┆ 1.2338e7    │
│ Canada                   ┆ 1.0038e7    │
│ United States of America ┆ 9.5099e6    │
│ China                    ┆ 9.4096e6    │
└──────────────────────────┴─────────────┘

 Průměrná rozloha státu podle kontinentů
shape: (8, 2)
┌─────────────────────────┬────────────────────────────┐
│ kontinent               ┆ prumerna_rozloha_statu_km2 │
│ ---                     ┆ ---                        │
│ str                     ┆ f64                        │
╞═════════════════════════╪════════════════════════════╡
│ Antarctica              ┆ 1.2338e7                   │
│ South America           ┆ 1.3663e6                   │
│ North America      

## Srovnání s klasickým postupem (přes Pandas)
Zde si ukážeme, jak by kod vypadal za použití kombinace GeoPandas / Pandas.

In [11]:
# 1. Úprava dat (kopie bez geometrie, přejmenování sloupců, výpočet km2)
df_pandas_alt = gdf_proj.drop(columns=["geometry"]).copy()
df_pandas_alt = df_pandas_alt.rename(columns={"NAME": "nazev_statu", "CONTINENT": "kontinent"})
df_pandas_alt["rozloha_km2"] = df_pandas_alt["rozloha_m2"] / 1_000_000

# 2. Výběr 5 největších zemí světa
nejvetsi_zeme_pd = df_pandas_alt.sort_values(by="rozloha_km2", ascending=False).head(5)
print("5 největších územních jednotek světa")
print(nejvetsi_zeme_pd[["nazev_statu", "rozloha_km2"]])

# 3. Porovnání průměrné rozlohy státu na kontinentech
kontinenty_pd = (
    df_pandas_alt.groupby("kontinent")["rozloha_km2"]
    .mean()
    .reset_index(name="prumerna_rozloha_statu_km2")
    .sort_values(by="prumerna_rozloha_statu_km2", ascending=False)
)

print("\n Průměrná rozloha státu podle kontinentů")
print(kontinenty_pd)

5 největších územních jednotek světa
                  nazev_statu   rozloha_km2
18                     Russia  1.702059e+07
159                Antarctica  1.233777e+07
3                      Canada  1.003773e+07
4    United States of America  9.509851e+06
139                     China  9.409565e+06

 Průměrná rozloha státu podle kontinentů
                 kontinent  prumerna_rozloha_statu_km2
1               Antarctica                1.233777e+07
7            South America                1.366306e+06
4            North America                1.360300e+06
5                  Oceania                1.214893e+06
2                     Asia                6.649055e+05
3                   Europe                5.914524e+05
0                   Africa                5.871584e+05
6  Seven seas (open ocean)                1.158961e+04
